In [1]:
# Запускается один раз перед началом работы
!pip install opencv-python mediapipe numpy

In [2]:
import cv2
import numpy as np

ORANGE_LOWER = np.array([5,  150, 150])
ORANGE_UPPER = np.array([20, 255, 255])

GREEN_LOWER  = np.array([35, 80,  80])
GREEN_UPPER  = np.array([85, 255, 255])


def get_color_mask(hsv_frame, lower, upper):
    mask = cv2.inRange(hsv_frame, lower, upper)
    kernel = np.ones((7, 7), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    return mask


def classify_shape(contour):
    epsilon = 0.03 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    vertices = len(approx)

    if vertices == 3:
        return 'triangle', approx
    elif vertices == 4:
        pts = approx.reshape(4, 2).astype(np.float32)

        sides = []
        for i in range(4):
            sides.append(np.linalg.norm(pts[i] - pts[(i+1)%4]))

        diag1 = np.linalg.norm(pts[0] - pts[2])
        diag2 = np.linalg.norm(pts[1] - pts[3])


        sides_ok = (max(sides) - min(sides)) / np.mean(sides) < 0.15

        diags_ok = abs(diag1 - diag2) / ((diag1 + diag2) / 2) < 0.15

        avg_side = np.mean(sides)
        avg_diag = (diag1 + diag2) / 2
        ratio_ok = abs(avg_diag / avg_side - 1.414) < 0.2  # Допуск ±20%

        if sides_ok and diags_ok and ratio_ok:
            return 'square', approx

    return None, approx


def draw_label(frame, text, position, color=(255, 255, 255), bg_color=(0, 0, 0)):
    x, y = position
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale, thickness = 0.7, 2
    (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
    cv2.rectangle(frame, (x - 4, y - th - 6), (x + tw + 4, y + 4), bg_color, -1)
    cv2.putText(frame, text, (x, y), font, scale, color, thickness)


print(" Вспомогательные функции Части 1 загружены.")

✅ Вспомогательные функции Части 1 загружены.


In [3]:
MIN_AREA = 3000

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print(" Камера не найдена. Попробуй изменить индекс: VideoCapture(1) или VideoCapture(2)")
else:
    print(" Камера открыта. Нажми Q или Esc для остановки.")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    mask_orange = get_color_mask(hsv, ORANGE_LOWER, ORANGE_UPPER)
    mask_green  = get_color_mask(hsv, GREEN_LOWER,  GREEN_UPPER)

    detected = []

    contours_o, _ = cv2.findContours(mask_orange, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours_o:
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        shape, approx = classify_shape(cnt)
        if shape == 'triangle':
            detected.append(('triangle', 'orange', approx))
            cv2.drawContours(frame, [approx], -1, (0, 165, 255), 3)
            x, y, _, _ = cv2.boundingRect(cnt)
            draw_label(frame, "ORANGE TRIANGLE", (x, y - 10),
                       color=(0, 165, 255), bg_color=(20, 20, 20))

    contours_g, _ = cv2.findContours(mask_green, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours_g:
        if cv2.contourArea(cnt) < MIN_AREA:
            continue
        shape, approx = classify_shape(cnt)
        if shape == 'square':
            detected.append(('square', 'green', approx))
            cv2.drawContours(frame, [approx], -1, (0, 255, 0), 3)
            x, y, _, _ = cv2.boundingRect(cnt)
            draw_label(frame, "GREEN SQUARE", (x, y - 10),
                       color=(0, 255, 0), bg_color=(20, 20, 20))


    has_triangle = any(d[0] == 'triangle' for d in detected)
    has_square   = any(d[0] == 'square'   for d in detected)

    if has_triangle and has_square:
        status_text  = "!! CRITICAL ERROR: INCOMPATIBLE !!"
        status_color = (0, 0, 255)
        bg            = (30, 0, 0)
    elif has_triangle:
        status_text  = "CAUTION: FLAMMABLE"
        status_color = (0, 165, 255)
        bg            = (20, 20, 0)
    elif has_square:
        status_text  = "SAFE: FOOD GRADE"
        status_color = (0, 255, 0)
        bg            = (0, 20, 0)
    else:
        status_text  = "SCANNING..."
        status_color = (200, 200, 200)
        bg            = (30, 30, 30)

    h, w = frame.shape[:2]
    (tw, th), _ = cv2.getTextSize(status_text, cv2.FONT_HERSHEY_DUPLEX, 1.1, 3)
    cv2.rectangle(frame, (0, 0), (w, th + 24), bg, -1)
    cv2.putText(frame, status_text, (w // 2 - tw // 2, th + 10),
                cv2.FONT_HERSHEY_DUPLEX, 1.1, status_color, 3)

    cv2.imshow("Variant 3 | Part 1: Hazard Classification (Q to quit)", frame)

    key = cv2.waitKey(1) & 0xFF
    if key in (ord('q'), ord('Q'), 27):   # Q или Esc
        break

cap.release()
cv2.destroyAllWindows()
print(" Камера закрыта.")

✅ Камера открыта. Нажми Q или Esc для остановки.
✅ Камера закрыта.


In [4]:
import urllib.request
import os

MODEL_PATH = "efficientdet.tflite"
MODEL_URL  = "https://storage.googleapis.com/mediapipe-models/object_detector/efficientdet_lite0/int8/1/efficientdet_lite0.tflite"

if not os.path.exists(MODEL_PATH):
    print("  Скачиваю модель EfficientDet (~5 MB)...")
    urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    print(" Модель сохранена:", MODEL_PATH)
else:
    print(" Модель уже скачана:", MODEL_PATH)

✅ Модель уже скачана: efficientdet.tflite


In [5]:
import cv2
import mediapipe as mp
import numpy as np

TARGET_CLASSES = {"cell phone", "laptop"}

BaseOptions    = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
DetectorOptions = mp.tasks.vision.ObjectDetectorOptions

options = DetectorOptions(
    base_options=BaseOptions(model_asset_path=MODEL_PATH),
    score_threshold=0.35,
    max_results=10
)

CLASS_CONFIG = {
    "cell phone": {
        "text":  "NO PHONES ALLOWED",
        "color": (0, 0, 255),
        "bg":    (0, 0, 120),
    },
    "laptop": {
        "text":  "LAPTOP DETECTED",
        "color": (0, 220, 255),
        "bg":    (0, 80, 100),
    },
}


def draw_detection(frame, bbox, label, score, cfg):
    x1 = int(bbox.origin_x)
    y1 = int(bbox.origin_y)
    x2 = int(bbox.origin_x + bbox.width)
    y2 = int(bbox.origin_y + bbox.height)

    cv2.rectangle(frame, (x1, y1), (x2, y2), cfg["color"], 3)

    tag = f"{label}: {score:.0%}"
    (tw, th), _ = cv2.getTextSize(tag, cv2.FONT_HERSHEY_SIMPLEX, 0.65, 2)
    cv2.rectangle(frame, (x1, y1 - th - 10), (x1 + tw + 8, y1), cfg["bg"], -1)
    cv2.putText(frame, tag, (x1 + 4, y1 - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.65, cfg["color"], 2)


cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print(" Камера не найдена. Попробуй VideoCapture(1)")
else:
    print(" Камера открыта. Нажми Q или Esc для остановки.")

with ObjectDetector.create_from_options(options) as detector:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)

        result = detector.detect(mp_image)

        active_warnings = set()
        for detection in result.detections:
            cat   = detection.categories[0]
            name  = cat.category_name.lower()
            if name not in TARGET_CLASSES:
                continue
            cfg = CLASS_CONFIG[name]
            draw_detection(frame, detection.bounding_box, name, cat.score, cfg)
            active_warnings.add(name)

        h, w = frame.shape[:2]
        if "cell phone" in active_warnings:
            status      = "!! NO PHONES ALLOWED !!"
            s_color     = (0, 0, 255)
            s_bg        = (0, 0, 60)
        elif "laptop" in active_warnings:
            status      = "LAPTOP DETECTED"
            s_color     = (0, 220, 255)
            s_bg        = (0, 50, 80)
        else:
            status      = "MONITORING..."
            s_color     = (180, 180, 180)
            s_bg        = (30, 30, 30)

        (tw, th), _ = cv2.getTextSize(status, cv2.FONT_HERSHEY_DUPLEX, 1.0, 2)
        cv2.rectangle(frame, (0, 0), (w, th + 22), s_bg, -1)
        cv2.putText(frame, status, (w // 2 - tw // 2, th + 8),
                    cv2.FONT_HERSHEY_DUPLEX, 1.0, s_color, 2)

        cv2.imshow("Variant 3 | Part 2: Device Detection (Q to quit)", frame)

        if cv2.waitKey(1) & 0xFF in (ord('q'), ord('Q'), 27):
            break

cap.release()
cv2.destroyAllWindows()
print(" Камера закрыта.")

✅ Камера открыта. Нажми Q или Esc для остановки.
✅ Камера закрыта.
